# RR Lyrae Template Fit on Fink/LSST Difference Alerts

This notebook demonstrates fitting a single RRab with `pycycle` using
Fink broker LSST prompt-product alerts from
`/astro/store/shire/pferguso/alert_sprint/concatenated_catalog/latest_obs.parquet`.

**Data notes:**
- One row per `diaObjectId`; the full light curve history is in `prvDiaSources`.
- We use `scienceFlux` / `scienceFluxErr` (direct science-image measurements) rather
  than the difference-image `psfFlux`, so the absolute magnitude carries physical meaning.
- Available bands: **g, r, i**.
- Template: DES (Long/Stringer `rr-templates`) with per-band LSST filter corrections.
- Model: `mag = β_b(P) + μ + E(B-V)·dust_b + A·γ_b(f·t + φ)`

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pycycle.templates import load_rr_template, RRTemplate
from pycycle.template_fit import TemplateFitter
from pycycle.lsdb_utils import flux_to_mag, apply_des_to_lsst_correction, compute_variability_features

PARQUET = '/astro/store/shire/pferguso/alert_sprint/concatenated_catalog/latest_obs.parquet'
TEMPLATE_DIR = os.path.expanduser('~/software/rr-templates/template_des')
BANDS = ['g', 'r', 'i']   # bands present in the Fink alert data

## 1. Load data and find a known RRab

In [ ]:
df = pd.read_parquet(PARQUET)
print(f'Catalog: {len(df)} objects')

# Objects with known VSX classification
def vsx_type(row):
    xm = row['xm']
    return xm.get('vsx_Type', None) if isinstance(xm, dict) else None

df['vsx_type'] = df.apply(vsx_type, axis=1)
rrab = df[df['vsx_type'].str.contains('RRAB', na=False)]
print(f'Known RRAB objects: {len(rrab)}')
print(rrab[['diaObjectId','vsx_type']].to_string(index=False))

In [ ]:
# Pick the object with the most observations for a clean demonstration
def n_good_obs(row):
    srcs = row['prvDiaSources']
    mag, magerr = flux_to_mag(
        [s['scienceFlux'] for s in srcs],
        [s['scienceFluxErr'] for s in srcs]
    )
    return int(np.sum((magerr > 0) & (magerr <= 0.2)))

rrab = rrab.copy()
rrab['n_good'] = rrab.apply(n_good_obs, axis=1)
target = rrab.sort_values('n_good', ascending=False).iloc[0]
print(f'Chosen object: diaObjectId = {target.diaObjectId}')
print(f'  VSX type : {target.vsx_type}')
print(f'  Good obs : {target.n_good}')

## 2. Parse the light curve

In [ ]:
srcs = target['prvDiaSources']
lc_raw = pd.DataFrame(list(srcs))[[
    'midpointMjdTai', 'band', 'scienceFlux', 'scienceFluxErr'
]]

# Convert Rubin nJy fluxes to AB magnitudes
mag, magerr = flux_to_mag(lc_raw['scienceFlux'].values, lc_raw['scienceFluxErr'].values)
lc_raw['mag']    = mag
lc_raw['magerr'] = magerr

# Quality cut: positive flux, uncertainty in (0, 0.2]
good = (lc_raw['magerr'] > 0) & (lc_raw['magerr'] <= 0.2) & lc_raw['band'].isin(BANDS)
lc = lc_raw[good].copy().reset_index(drop=True)

print(f'Total detections : {len(lc_raw)}')
print(f'After quality cut: {len(lc)}')
print()
print(lc.groupby('band').agg(
    n=('mag','count'),
    mag_mean=('mag','mean'),
    mag_std=('mag','std'),
    magerr_median=('magerr','median'),
).round(3))
print(f'\nTimespan: {lc["midpointMjdTai"].max() - lc["midpointMjdTai"].min():.2f} days')

In [ ]:
# --- Raw light curve ---
band_colors = {'g': '#2ca02c', 'r': '#d62728', 'i': '#ff7f0e'}

fig, ax = plt.subplots(figsize=(12, 4))
for band, sub in lc.groupby('band'):
    ax.errorbar(
        sub['midpointMjdTai'], sub['mag'], yerr=sub['magerr'],
        fmt='o', ms=3, lw=0.5, color=band_colors[band], label=band, alpha=0.7
    )
ax.invert_yaxis()
ax.set_xlabel('MJD (TAI)')
ax.set_ylabel('mag (AB)')
ax.set_title(f'Light curve — diaObjectId {target.diaObjectId} ({target.vsx_type})')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Variability pre-filter check

Following Stringer et al. (2019) §3.2: keep objects where
`lchi_med ≥ 0.5` **and** `sig_max ≥ 0`.

In [ ]:
feats = compute_variability_features(lc, BANDS)
passes = feats['lchi_med'] >= 0.5 and feats['sig_max'] >= 0.0
print(f"lchi_med = {feats['lchi_med']:.3f}  (threshold: ≥ 0.5)")
print(f"sig_max  = {feats['sig_max']:.3f}  (threshold: ≥ 0.0)")
print(f"Passes variability filter: {passes}")

## 4. Load DES template (gri subset) with LSST filter correction

In [ ]:
# Load the full DES template (g,i,r,Y,z)
template_des = load_rr_template(TEMPLATE_DIR, name='des')

# Apply DES → LSST zero-point correction at load time (zero runtime cost)
apply_des_to_lsst_correction(template_des)

# Extract gri subset — only bands present in the data
sub_idx = [template_des.bands.index(b) for b in BANDS]
template_gri = RRTemplate(
    name='des_gri',
    bands=BANDS,
    phase=template_des.phase,
    gamma=template_des.gamma[sub_idx],
    dust=template_des.dust[sub_idx],
    betas=template_des.betas[sub_idx],
)
print('Template:', template_gri.name)
print('Bands   :', template_gri.bands)
print('Dust (A_b / A_V):', dict(zip(template_gri.bands, template_gri.dust.round(3))))
print('Betas (c0, p1, p2) per band:')
for b, beta in zip(template_gri.bands, template_gri.betas):
    print(f'  {b}: {beta.round(4)}')

## 5. Template fit

We use `warm_start=True` which carries the best solution
(φ, μ, E(B-V), A) from one trial frequency to the next — ~4× faster than
trying `n_start` random phases at each frequency.

Period search range: **0.44 – 0.89 d** (RRab only).

In [ ]:
# Build input arrays
band_map = {b: i for i, b in enumerate(BANDS)}
t_arr  = lc['midpointMjdTai'].values
m_arr  = lc['mag'].values
me_arr = lc['magerr'].values
bi_arr = lc['band'].map(band_map).astype(int).values

# Fit
fitter = TemplateFitter(template_gri, n_newton=5, warm_start=True)
t0 = time.perf_counter()
result = fitter.fit(t_arr, m_arr, me_arr, bi_arr, BANDS, pmin=0.44, dphi=0.02, pmax=0.89)
elapsed = time.perf_counter() - t0

print(f'Fit complete in {elapsed:.2f} s  ({len(result.periods)} trial periods)')
print(f'Best period : {result.best_period:.6f} days')
print(f'Best coeffs : {result.best_coeffs}')
print()
print('Top 5 candidate periods:')
for i, cand in enumerate(result.top_periods(5)):
    print(f"  {i+1}. P = {cand['period']:.6f} d   RSS = {cand['rss']:.4f}")

In [ ]:
# RSS vs period
result.plot_rss()
plt.title(f'RSS vs period — diaObjectId {target.diaObjectId}')
plt.tight_layout()
plt.show()

In [ ]:
# Phased light curve with template overlay
result.plot_phased()
plt.suptitle(
    f'diaObjectId {target.diaObjectId} ({target.vsx_type})\n'
    f'P = {result.best_period:.6f} d   '
    f'μ = {result.best_coeffs["mu"]:.2f}   '
    f'E(B-V) = {result.best_coeffs["EBV"]:.3f}   '
    f'A = {result.best_coeffs["A"]:.3f}',
    fontsize=10, y=1.01
)
plt.tight_layout()
plt.show()

## 6. Interpret the fit parameters

The rr-templates model gives physically meaningful outputs:

| Parameter | Symbol | Value | Meaning |
|---|---|---|---|
| Distance modulus | μ | see above | Distance d = 10^((μ−10)/5) kpc |
| Dust extinction | E(B-V) | see above | Should be ≥ 0 for a physical fit |
| Amplitude | A | see above | Peak-to-trough variation (0.3–1.5 for RRab) |
| Period | P | see above | 0.44–0.89 d for RRab |

**Quality flags to check before accepting a fit:**
- `E(B-V)` should be ≥ 0 (negative = bad fit or insufficient band coverage)
- `A` in [0.3, 1.5] for normal RRab
- RSS / (N_obs − 4) should be close to 1 (normalised residual)
- Best period should be unambiguous in the RSS plot (clear global minimum)

In [ ]:
mu  = result.best_coeffs['mu']
EBV = result.best_coeffs['EBV']
A   = result.best_coeffs['A']
n_params = 4  # mu, EBV, A, phi
rss_dof = result.rss.min() / max(len(lc) - n_params, 1)

distance_kpc = 10 ** ((mu - 10) / 5)
print(f'Period       : {result.best_period:.6f} days')
print(f'Dist. modulus: μ = {mu:.3f}  →  d ≈ {distance_kpc:.1f} kpc')
print(f'Extinction   : E(B-V) = {EBV:.3f}  ({"OK" if EBV >= 0 else "NEGATIVE — suspect fit"})')
print(f'Amplitude    : A = {A:.3f}  ({"OK" if 0.3 <= A <= 1.5 else "outside normal RRab range"})')
print(f'RSS/dof      : {rss_dof:.4f}  ({"OK" if rss_dof < 2 else "large — check fit quality"})')